In [1]:
import pandas as pd
import tensorflow as tf
print(tf.__version__)

2.21.0


In [2]:
dataset = pd.read_csv('Churn_Modelling.csv')
print(dataset.shape)
dataset.head()

(10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Check for missing values
print(dataset.isnull().sum().sum(), "missing values\n")

# Churn rate for each category (0.20 = 20% of that group left)
for col in ['Geography', 'Gender', 'IsActiveMember', 'NumOfProducts', 'HasCrCard']:
    print(dataset.groupby(col)['Exited'].mean().round(3), '\n')

# Correlation of numeric features with Exited
numeric = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'Exited']
print(dataset[numeric].corr()['Exited'].sort_values())

0 missing values

Geography
France     0.162
Germany    0.324
Spain      0.167
Name: Exited, dtype: float64 

Gender
Female    0.251
Male      0.165
Name: Exited, dtype: float64 

IsActiveMember
0    0.269
1    0.143
Name: Exited, dtype: float64 

NumOfProducts
1    0.277
2    0.076
3    0.827
4    1.000
Name: Exited, dtype: float64 

HasCrCard
0    0.208
1    0.202
Name: Exited, dtype: float64 

CreditScore       -0.027094
Tenure            -0.014001
EstimatedSalary    0.012097
Balance            0.118533
Age                0.285323
Exited             1.000000
Name: Exited, dtype: float64


In [4]:
X = dataset.iloc[:, 3:-1].values   # CreditScore through EstimatedSalary (10 features)
y = dataset.iloc[:, -1].values     # Exited
print(X[0])

[619 'France' 'Female' 42 2 0.0 1 1 1 101348.88]


In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])
print(X[0])

[619 'France' 0 42 2 0.0 1 1 1 101348.88]


In [7]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])],
                       remainder='passthrough')
X = np.array(ct.fit_transform(X))
print(X[0])

[1.0 0.0 0.0 619 0 42 2 0.0 1 1 1 101348.88]


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=29, stratify=y
)
print(X_train.shape, X_test.shape)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

(8000, 12) (2000, 12)


In [9]:
tf.keras.utils.set_random_seed(29)

ann = tf.keras.models.Sequential([
    tf.keras.Input(shape=(12,)),                          # 12 features in
    tf.keras.layers.Dense(units=6, activation='relu'),    # hidden layer 1
    tf.keras.layers.Dense(units=6, activation='relu'),    # hidden layer 2
    tf.keras.layers.Dense(units=1, activation='sigmoid')  # output layer
])

ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
ann.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 6)              │            78 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │            42 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             7 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 127 (508.00 B)

 Trainable params: 127 (508.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
ann.fit(X_train, y_train, batch_size=32, epochs=100)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 542us/step - accuracy: 0.7691 - loss: 0.5356
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 570us/step - accuracy: 0.7962 - loss: 0.4592
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 587us/step - accuracy: 0.7962 - loss: 0.4454
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 633us/step - accuracy: 0.7962 - loss: 0.4372
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 587us/step - accuracy: 0.7958 - loss: 0.4314
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 554us/step - accuracy: 0.7975 - loss: 0.4267
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 546us/step - accuracy: 0.8083 - loss: 0.4218
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 535us/step - accuracy: 0.8209 - loss: 0.4149
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 513us/step - accuracy: 0.8267 - loss: 0.4066
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 527us/step - accuracy: 0.8309 - loss: 0.3988
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 558us/step - accuracy: 0.8338 - loss: 0.3927
Epoch 12/100
250/25

In [12]:
from sklearn.metrics import confusion_matrix, accuracy_score

y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)

cm = confusion_matrix(y_test, y_pred)
print(cm)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step
[[1513   80]
 [ 221  186]]
Accuracy: 0.8495


In [13]:
# Average profile: customers who stayed (0) vs. left (1)
profile = dataset.groupby('Exited')[['CreditScore', 'Age', 'Tenure', 'Balance',
                                     'NumOfProducts', 'IsActiveMember', 'EstimatedSalary']].mean()
profile.round(2)

,CreditScore,Age,Tenure,Balance,NumOfProducts,IsActiveMember,EstimatedSalary
Exited,,,,,,,
0,651.85,37.41,5.03,72745.30,1.54,0.55,99738.39
1,645.35,44.84,4.93,91108.54,1.48,0.36,101465.68


In [14]:
cols = profile.columns
effect = (profile.loc[1] - profile.loc[0]) / dataset[cols].std()
effect.round(2).sort_values()

IsActiveMember    -0.39
NumOfProducts     -0.12
CreditScore       -0.07
Tenure            -0.03
EstimatedSalary    0.03
Balance            0.29
Age                0.71
dtype: float64